In [6]:
import pandas as pd
from datetime import datetime

# convert date --> datetime
def convert_datetime(dt):
    date_part = str(dt).split()[0]
    return datetime.strptime(date_part, "%d/%m/%Y")

# convert household_size --> number
def convert_household_size(x):
    if x in [str(i) for i in range(1, 8)]:
        return int(x)
    elif x == "8 or more":
        return 8
    elif x in ["Prefer not to say", "Don't know"]:
        return None
    return x

# read raw data
df = pd.read_csv(
    "../data/raw/australia.csv",
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

# convert date column
df["endtime"] = df["endtime"].apply(convert_datetime)

# delete columns with too many missing values
missing_counts = df.isnull().sum()
threshold = 10781
drop_cols = missing_counts[missing_counts > threshold].index.tolist()
# delete
df = df.drop(columns=drop_cols)

# enter N/A for medical questions that were not authorized to be answered within a specific time period
start_date = pd.to_datetime("2021-02-10")
end_date = pd.to_datetime("2021-10-18")
mask = (df["endtime"] >= start_date) & (df["endtime"] <= end_date)

for i in range(1, 5):
    df.loc[mask, f"PHQ4_{i}"] = df.loc[mask, f"PHQ4_{i}"].fillna("N/A")

for i in range(1, 14):
    df.loc[mask, f"d1_health_{i}"] = df.loc[mask, f"d1_health_{i}"].fillna("N/A")

for i in range(98, 100):
    df.loc[mask, f"d1_health_{i}"] = df.loc[mask, f"d1_health_{i}"].fillna("N/A")

# delete remaining missing values
df = df.dropna()

# convert r1_1 and r1_2 --> numerical scales
scale_map = {
    "7 - Agree": 7,
    "6": 6,
    "5": 5,
    "4": 4,
    "3": 3,
    "2": 2,
    "1 – Disagree": 1
}

for i in range(1, 3):
    df[f"r1_{i}"] = pd.to_numeric(
    df[f"r1_{i}"].map(scale_map),
    errors="coerce"
)

# convert star with "i12_health_" --> numerical values
frequency_map = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

for col in df.columns:
    if col.startswith("i12_health_"):
        df[col] = df[col].map(frequency_map)

# create mask behavior variables
mask_cols = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
df["face_mask_behaviour_scale"] = df[mask_cols].median(axis=1)
df["face_mask_behaviour_binary"] = df["face_mask_behaviour_scale"].apply(
    lambda x: "Yes" if x >= 4 else "No"
)

# create overall protection behavior variables
protective_cols = [col for col in df.columns if col.startswith("i12_")]
df["protective_behaviour_scale"] = df[protective_cols].median(axis=1)
df["protective_behaviour_binary"] = df["protective_behaviour_scale"].apply(
    lambda x: "Yes" if x >= 4 else "No"
)

# create protective behavior variables without masks
protective_nomask_cols = [col for col in protective_cols if col not in mask_cols]
df["protective_behaviour_nomask_scale"] = df[protective_nomask_cols].median(axis=1)

# comorbidity information
d1_cols = [col for col in df.columns if col.startswith("d1_")]
df["d1_comorbidities"] = "Yes"
df.loc[df["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"
df.loc[df["d1_health_99"] == "N/A", "d1_comorbidities"] = "NA"
df.loc[df["d1_health_98"] == "Yes", "d1_comorbidities"] = "Prefer_not_to_say"
df = df.drop(columns=d1_cols)

# regenerate week_number (each period is 14 days)
start_time = df["endtime"].min()
df["week_number"] = ((df["endtime"] - start_time).dt.days // 14) + 1

# handling household_size
df["household_size"] = df["household_size"].apply(convert_household_size)
df = df.dropna()

# delete unnecessary columns
df = df.drop(columns=["qweek", "weight"] + protective_cols)

# save cleaned data
df.to_csv("../data/processed/cleaned_data.csv", index=False)